### Instituto Brasileiro de Geografia e Estatística (IBGE)

Populacao municipal IBGE para o projeto COVID-19 (2020–2024).

No notebook:
    %pip install pandas pyarrow xlrd
    %run baixar_populacao_ibge_2020_2024.py

Saida: dados_ibge/populacao_municipios_2020_2024.parquet
Cache: dados_ibge/fontes/ (respostas originais e XLS oficial).

2020/2021/2024: SIDRA 6579, variavel 9324, estimativas em 1 de julho.
2022: SIDRA 4714, variavel 93, Censo 2022.
2023: relacao publicada no DOU em 2023, baseada no Censo 2022 com malha 2023.
      NAO e uma estimativa demografica de 2023. ano_referencia_populacao=2022.
O ano de analise serve para organizar a serie, sem ocultar esta diferenca.
As edicoes publicadas tem metodologias/malhas distintas; variacoes entre elas
nao devem ser interpretadas diretamente como crescimento demografico.

Um registro por codigo municipal de 7 digitos e ano_analise. Codigo de 6
 digitos e o codigo de 7 sem o ultimo digito, para comparacao com DATASUS.
Confirmar formato e residencia/ocorrencia antes de juntar ao SRAG. Populacao
nao deve ser somada apos replicacao em linhas de notificacoes. Agregar casos
por municipio/ano e depois cruzar. Para taxas por idade/sexo sao necessarios
denominadores especificos, nao esta populacao municipal total.

In [1]:
import io
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_ibge'
CACHE = PASTA / 'fontes'
ATUALIZAR_FONTES = False  # True: baixa novamente; False: reutiliza cache.
URL_EST = 'https://apisidra.ibge.gov.br/values/t/6579/n6/all/v/9324/p/2020,2021,2024'
URL_CENSO = 'https://apisidra.ibge.gov.br/values/t/4714/n6/all/v/93/p/2022'
URL_2023 = ('https://ftp.ibge.gov.br/Informacoes_Gerais_e_Referencia/'
    'Relacao_da_Populacao_dos_Municipios_para_publicacao_no_DOU_em_2023/'
    'POP_DOU_2023_Municipios_POP2022_Malha2023.xls')
UF = {'11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
      '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA',
      '31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS',
      '50':'MS','51':'MT','52':'GO','53':'DF'}

In [3]:
def obter(url, nome):
    caminho = CACHE / nome
    if caminho.exists() and not ATUALIZAR_FONTES:
        print('Reutilizando: ' + nome, flush=True)
        return caminho.read_bytes()
    for tentativa in range(1, 4):
        try:
            print(f'Baixando {nome} (tentativa {tentativa}/3)...', flush=True)
            with urlopen(Request(url, headers={'User-Agent':'Observatorio-COVID-Brasil/1.0'}), timeout=120) as r:
                conteudo = r.read()
            if nome.endswith('.json'):
                valor = json.loads(conteudo)
                if not isinstance(valor, list) or len(valor)<2 or 'V' not in valor[0]:
                    raise ValueError('SIDRA nao retornou uma tabela com registros.')
            elif not conteudo.startswith(bytes.fromhex('D0CF11E0A1B11AE1')):
                raise ValueError('Resposta nao e o XLS esperado.')
            parcial = caminho.with_suffix(caminho.suffix + '.part')
            parcial.write_bytes(conteudo)
            parcial.replace(caminho)
            (CACHE / (nome + '.meta.json')).write_text(json.dumps({
                'url':url, 'coleta_utc':datetime.now(timezone.utc).isoformat(),
                'bytes':len(conteudo)},ensure_ascii=False,indent=2),encoding='utf-8')
            return conteudo
        except Exception as erro:
            print(f'Falha: {erro}',flush=True)
            if tentativa == 3:
                raise
            time.sleep(2*tentativa)


def populacao(valor):
    s = str(valor).strip()
    # Simbolos especiais do SIDRA: mantidos em valor_original, nunca como zero.
    if s in {'-', '...', '..', '.', 'X', '', 'nan'}:
        return None
    if not re.fullmatch(r'\d+',s):
        raise ValueError(f'Populacao em formato nao previsto: {s!r}')
    return int(s)


def linha(codigo, municipio, ano, valor, tipo, ref, data_ref, url, tabela):
    codigo = str(codigo).strip()
    if not re.fullmatch(r'\d{7}',codigo) or codigo[:2] not in UF:
        raise ValueError(f'Codigo IBGE invalido: {codigo}')
    pop = populacao(valor)
    return {'ano_analise':ano, 'codigo_ibge_7':codigo, 'codigo_ibge_6':codigo[:6],
            'codigo_uf':codigo[:2], 'uf':UF[codigo[:2]], 'municipio':municipio,
            'populacao':pop, 'valor_original':str(valor),
            'status_populacao':'disponivel' if pop is not None else 'indisponivel_na_fonte',
            'tipo_fonte':tipo, 'ano_referencia_populacao':ref,
            'data_referencia_populacao':data_ref,
            'exige_cautela_temporal':ano==2023, 'tabela_fonte':tabela, 'url_fonte':url}


def ler_sidra(conteudo, url, censo=False):
    dados = json.loads(conteudo)
    cab = dados[0]
    def coluna(rotulo):
        encontrados = [k for k,v in cab.items() if v==rotulo]
        if len(encontrados)!=1:
            raise ValueError(f'Cabecalho SIDRA inesperado: {rotulo}')
        return encontrados[0]
    cod = coluna('Município (Código)')
    nome = coluna('Município')
    ano = coluna('Ano (Código)')
    var = coluna('Variável (Código)')
    linhas = []
    for r in dados[1:]:
        a = int(r[ano])
        if r[var] != ('93' if censo else '9324'):
            raise ValueError('Variavel diferente de populacao na resposta.')
        if a not in ({2022} if censo else {2020,2021,2024}):
            raise ValueError(f'Ano inesperado: {a}')
        nm = re.sub(r'\s*(?:-\s*[A-Z]{2}|\([A-Z]{2}\))$', '',r[nome]).strip()
        linhas.append(linha(r[cod],nm,a,r['V'],'Censo 2022' if censo else 'Estimativa municipal',
                            a,f'{a}-08-01' if censo else f'{a}-07-01',url,'4714' if censo else '6579'))
    return linhas


def ler_2023(conteudo):
    x = pd.read_excel(io.BytesIO(conteudo),header=1,dtype=str,engine='xlrd')
    x.columns = [str(c).strip() for c in x.columns]
    exigidas = {'UF','COD. UF','COD. MUNIC','NOME DO MUNICÍPIO','POPULAÇÃO'}
    if not exigidas.issubset(x.columns):
        raise ValueError('Estrutura do XLS 2023 mudou: ' + str(list(x.columns)))
    x = x[x['UF'].isin(UF.values())]
    linhas = []
    for _,r in x.iterrows():
        codigo = r['COD. UF'].strip().zfill(2) + r['COD. MUNIC'].strip().zfill(5)
        linhas.append(linha(codigo,r['NOME DO MUNICÍPIO'],2023,r['POPULAÇÃO'],
            'Censo 2022 - divulgacao DOU 2023 / malha 2023',2022,'2022-08-01',URL_2023,'DOU 2023'))
    if len(linhas)<5000:
        raise ValueError('Leitura incompleta do arquivo municipal 2023.')
    return linhas


def main():
    CACHE.mkdir(parents=True,exist_ok=True)
    dados = ler_sidra(obter(URL_EST,'sidra_estimativas.json'),URL_EST)
    dados += ler_sidra(obter(URL_CENSO,'sidra_censo_2022.json'),URL_CENSO,censo=True)
    dados += ler_2023(obter(URL_2023,'populacao_dou_2023.xls'))
    df = pd.DataFrame(dados)
    df['populacao'] = pd.array(df['populacao'],dtype='Int64')
    if set(df['ano_analise']) != set(range(2020,2025)):
        raise ValueError('Faltam anos na coleta.')
    if df.duplicated(['ano_analise','codigo_ibge_7']).any():
        raise ValueError('Chaves municipio/ano duplicadas.')
    if df.duplicated(['ano_analise','codigo_ibge_6']).any():
        raise ValueError('Colisao nos codigos de 6 digitos; conferir antes de cruzar com SRAG.')
    if (df['populacao'].dropna()<=0).any():
        raise ValueError('Populacao nao positiva; verificar fonte.')
    contagens = df.groupby('ano_analise').size()
    if (contagens<5000).any():
        raise ValueError('Quantidade de municipios inesperadamente baixa.')
    df = df.sort_values(['ano_analise','codigo_ibge_7']).reset_index(drop=True)
    destino = PASTA / 'populacao_municipios_2020_2024.parquet'
    parcial = destino.with_suffix('.parquet.part')
    df.to_parquet(parcial,index=False,engine='pyarrow',compression='snappy')
    parcial.replace(destino)
    resumo = df.groupby('ano_analise').agg(registros=('codigo_ibge_7','size'),
        populacoes_disponiveis=('populacao','count'),soma_valores_disponiveis=('populacao','sum'))
    print('\n',resumo.to_string())
    print('\n2023 usa populacao censitaria de 2022, com malha territorial de 2023.')
    print('Simbolos de indisponibilidade da fonte foram preservados e a populacao ficou nula.')
    print(f'\nArquivo salvo: {destino}')
    (PASTA/'LEIA_ME.txt').write_text(__doc__ + '\n\n' + resumo.to_string() +
        '\n\nO arquivo DOU 2023 possui nota sobre ajustes territoriais posteriores em Vitoria de Santo Antao e Pombos (PE).\n'
        'Foi usada a edicao DOU indicada na URL, sem substituir por outras edicoes.\n',encoding='utf-8')
    return df


if __name__ == '__main__':
    populacao_ibge = main()

Baixando sidra_estimativas.json (tentativa 1/3)...
Baixando sidra_censo_2022.json (tentativa 1/3)...
Baixando populacao_dou_2023.xls (tentativa 1/3)...

              registros  populacoes_disponiveis  soma_valores_disponiveis
ano_analise                                                             
2020              5571                    5570                 211755692
2021              5571                    5570                 213317639
2022              5570                    5570                 203080756
2023              5570                    5570                 203080756
2024              5571                    5570                 212583750

2023 usa populacao censitaria de 2022, com malha territorial de 2023.
Simbolos de indisponibilidade da fonte foram preservados e a populacao ficou nula.

Arquivo salvo: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_ibge\populacao_municipios_2020_2024.parquet


### Análise exploratória

Analise exploratoria da populacao municipal IBGE ja baixada.

No notebook:
    %pip install pandas pyarrow matplotlib
    %run analisar_populacao_ibge_2020_2024.py

Entrada: dados_ibge/populacao_municipios_2020_2024.parquet
Saida: dados_ibge/analise_populacao/relatorio_populacao_ibge.html
       tabelas CSV e Parquet; graficos PNG.
Nao baixa dados, nao altera a fonte e nao cruza ainda com o SRAG.
Totais sao somas dos valores municipais disponiveis por ano, nunca a soma
entre anos. 2023 usa Censo 2022 / malha 2023. Comparacoes entre publicacoes
nao sao medidas diretas de crescimento demografico.

In [4]:
import base64
import html
import math
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [5]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ENTRADA = BASE / 'dados_ibge' / 'populacao_municipios_2020_2024.parquet'
SAIDA = BASE / 'dados_ibge' / 'analise_populacao'
ANO_DESTAQUE = 2024
UF_COD = {'11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
          '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA',
          '31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS',
          '50':'MS','51':'MT','52':'GO','53':'DF'}
REGIOES = {uf: regiao for regiao,ufs in {
    'Norte':'AC AP AM PA RO RR TO', 'Nordeste':'AL BA CE MA PB PE PI RN SE',
    'Sudeste':'ES MG RJ SP', 'Sul':'PR RS SC', 'Centro-Oeste':'DF GO MT MS'
}.items() for uf in ufs.split()}

In [6]:
def numero(x):
    if pd.isna(x):
        return '—'
    if isinstance(x, (float,)) and not x.is_integer():
        return f'{x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    if isinstance(x, (int, float)):
        return f'{int(x):,}'.replace(',', '.')
    return str(x)


def exportar(df, nome):
    df.to_csv(SAIDA / f'{nome}.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
    df.to_parquet(SAIDA / f'{nome}.parquet', index=False)


def tabela(titulo, df, limite=35):
    vista = df.head(limite).copy()
    for c in vista.columns:
        if pd.api.types.is_numeric_dtype(vista[c]):
            vista[c] = vista[c].map(numero)
    extra = f'<p>Exibindo {limite} de {len(df)} linhas. Consulte a tabela completa em CSV ou Parquet.</p>' if len(df)>limite else ''
    return '<section><h2>' + html.escape(titulo) + '</h2><div class="scroll">' + vista.to_html(index=False,escape=True,border=0,na_rep='—') + '</div>' + extra + '</section>'


def grafico(fig, nome, legenda):
    arquivo = SAIDA / f'{nome}.png'
    fig.tight_layout()
    fig.savefig(arquivo, dpi=140, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    imagem = base64.b64encode(arquivo.read_bytes()).decode('ascii')
    return f'<section><img alt="{html.escape(legenda)}" src="data:image/png;base64,{imagem}"><p>{html.escape(legenda)}</p></section>'


def agregado(df, grupos):
    # df contem apenas chaves e populacoes validas (populacao nula permitida).
    g = df.groupby(grupos, dropna=False)
    resultado = g.agg(registros=('codigo_ibge_7','size'),
        municipios_com_populacao=('populacao','count'),
        soma_populacao_disponivel=('populacao',lambda s:s.sum(min_count=1)),
        menor_populacao=('populacao','min'), mediana_municipal=('populacao','median'),
        media_municipal=('populacao','mean'), maior_populacao=('populacao','max')).reset_index()
    resultado['registros_sem_populacao'] = resultado['registros'] - resultado['municipios_com_populacao']
    return resultado


def main():
    if not ENTRADA.is_file():
        raise FileNotFoundError(f'Arquivo nao encontrado: {ENTRADA}. Execute primeiro o script de download.')
    SAIDA.mkdir(parents=True,exist_ok=True)
    print('1/4 — Lendo e conferindo a base...',flush=True)
    df = pd.read_parquet(ENTRADA)
    requeridas = {'ano_analise','codigo_ibge_7','codigo_ibge_6','uf','municipio','populacao',
                  'tipo_fonte','ano_referencia_populacao','data_referencia_populacao'}
    if not requeridas.issubset(df.columns):
        raise ValueError('Colunas ausentes: ' + ', '.join(sorted(requeridas-set(df.columns))))
    if df.empty:
        raise ValueError('A base esta vazia.')
    perfil = pd.DataFrame({'coluna':df.columns, 'tipo':[str(t) for t in df.dtypes],
        'nulos':[int(df[c].isna().sum()) for c in df],
        'valores_distintos_sem_nulos':[int(df[c].nunique()) for c in df]})
    perfil['percentual_nulos'] = (100*perfil['nulos']/len(df)).round(2)
    exportar(perfil,'perfil_colunas')
    for c in ['codigo_ibge_7','codigo_ibge_6','uf','municipio']:
        df[c] = df[c].astype('string').str.strip()
    df['uf'] = df['uf'].str.upper()
    pop = pd.to_numeric(df['populacao'],errors='coerce')
    ano = pd.to_numeric(df['ano_analise'],errors='coerce')
    problemas = []
    bloqueios = []
    def registrar(mascara, motivo, bloquear=True):
        mascara = mascara.fillna(True)
        if mascara.any():
            x = df.loc[mascara].copy()
            x['problema'] = motivo
            problemas.append(x)
            if bloquear:
                bloqueios.append(motivo)
    registrar(~df['codigo_ibge_7'].str.fullmatch(r'\d{7}',na=False),'Codigo municipal de 7 digitos invalido')
    registrar(~df['codigo_ibge_6'].str.fullmatch(r'\d{6}',na=False) |
              df['codigo_ibge_6'].ne(df['codigo_ibge_7'].str[:6]),'Codigo de 6 digitos inconsistente')
    registrar(~df['uf'].isin(REGIOES) | df['uf'].ne(df['codigo_ibge_7'].str[:2].map(UF_COD)), 'UF inconsistente com codigo municipal')
    registrar(~ano.isin(range(2020,2025)), 'Ano fora de 2020–2024 ou invalido')
    registrar(df.duplicated(['ano_analise','codigo_ibge_7'],keep=False), 'Chave municipio/ano repetida')
    registrar(df.duplicated(['ano_analise','codigo_ibge_6'],keep=False), 'Chave de 6 digitos/ano repetida')
    pop_invalida = (df['populacao'].notna() & pop.isna()) | (pop.notna() &
        (~pop.map(lambda x: math.isfinite(float(x)) if pd.notna(x) else True) | (pop<=0) | (pop%1!=0)))
    registrar(pop_invalida,'Populacao nao numerica, nao inteira ou nao positiva')
    registrar(pop.isna() & df['populacao'].isna(),'Populacao ausente na fonte',bloquear=False)
    diagnostico = pd.concat(problemas,ignore_index=True) if problemas else df.iloc[:0].assign(problema=pd.Series(dtype='string'))
    exportar(diagnostico,'registros_com_alertas')
    if bloqueios:
        raise ValueError('Corrija os problemas estruturais antes de agregar: ' + '; '.join(sorted(set(bloqueios))) +
                         f'. Detalhes em {SAIDA / "registros_com_alertas.csv"}')
    df['ano_analise'] = ano.astype(int)
    df['populacao'] = pop.astype('Int64')
    df['regiao'] = df['uf'].map(REGIOES)
    ausentes = sorted(set(range(2020,2025))-set(df['ano_analise']))
    if ausentes:
        raise ValueError(f'Faltam anos inteiros: {ausentes}')
    print('2/4 — Calculando resumos por ano, UF e regiao...',flush=True)
    anual = agregado(df,['ano_analise'])
    uf = agregado(df,['ano_analise','uf'])
    regiao = agregado(df,['ano_analise','regiao'])
    fontes = df.groupby(['ano_analise','tipo_fonte','ano_referencia_populacao',
                         'data_referencia_populacao'],dropna=False).size().reset_index(name='registros')
    for nome,dados in [('resumo_anual',anual),('resumo_uf',uf),('resumo_regiao',regiao),('fontes_por_ano',fontes)]:
        exportar(dados,nome)
    # Somente cidades com populacao conhecida participam dos rankings.
    validos = df[df['populacao'].notna()].copy()
    ordem = ['ano_analise','populacao','codigo_ibge_7']
    ranking = validos.sort_values(ordem,ascending=[True,False,True]).copy()
    ranking['posicao_no_ano'] = ranking.groupby('ano_analise').cumcount()+1
    totais = validos.groupby('ano_analise')['populacao'].sum()
    ranking['participacao_percentual_no_ano'] = 100*ranking['populacao']/ranking['ano_analise'].map(totais)
    campos = ['ano_analise','posicao_no_ano','codigo_ibge_7','municipio','uf','regiao','populacao','participacao_percentual_no_ano']
    exportar(ranking[campos],'ranking_municipios')
    top = ranking.groupby('ano_analise',sort=False).head(20)[campos]
    menores = validos.sort_values(ordem,ascending=[True,True,True]).groupby('ano_analise').head(20)
    exportar(top,'maiores_20_por_ano')
    exportar(menores[['ano_analise','codigo_ibge_7','municipio','uf','populacao']],'menores_20_por_ano')
    limites = [0,5000,10000,20000,50000,100000,500000,float('inf')]
    rotulos = ['Ate 5 mil','Mais de 5 a 10 mil','Mais de 10 a 20 mil','Mais de 20 a 50 mil',
               'Mais de 50 a 100 mil','Mais de 100 a 500 mil','Mais de 500 mil']
    validos['porte'] = pd.cut(validos['populacao'].astype(float),bins=limites,labels=rotulos)
    porte = validos.groupby(['ano_analise','porte'],observed=True).agg(
        municipios=('codigo_ibge_7','size'),populacao=('populacao','sum')).reset_index()
    porte['percentual_municipios_validos'] = 100*porte['municipios']/porte.groupby('ano_analise')['municipios'].transform('sum')
    porte['percentual_populacao_disponivel'] = 100*porte['populacao']/porte.groupby('ano_analise')['populacao'].transform('sum')
    porte['porte'] = porte['porte'].astype('string')
    exportar(porte,'distribuicao_por_porte')
    concentracao = []
    for a,g in ranking.groupby('ano_analise'):
        for n in [10,20,100]:
            concentracao.append({'ano_analise':a,'grupo':f'Top {n}', 'municipios_no_grupo':min(n,len(g)),
                'populacao_grupo':int(g.head(n)['populacao'].sum()),
                'percentual_populacao_disponivel':float(100*g.head(n)['populacao'].sum()/totais.loc[a])})
    concentracao = pd.DataFrame(concentracao)
    exportar(concentracao,'concentracao_populacional')
    # Valores publicados lado a lado: sem rotular diferencas como crescimento.
    comparativo = df.pivot(index='codigo_ibge_7',columns='ano_analise',values='populacao')
    comparativo.columns = ['populacao_publicada_'+str(a) for a in comparativo.columns]
    nomes = df.sort_values('ano_analise').drop_duplicates('codigo_ibge_7',keep='last')[['codigo_ibge_7','municipio','uf']]
    comparativo = nomes.merge(comparativo.reset_index(),on='codigo_ibge_7',validate='one_to_one')
    exportar(comparativo,'comparativo_publicacoes')
    print('3/4 — Gerando graficos...',flush=True)
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False})
    imagens=[]
    fig,ax=plt.subplots(figsize=(9,4))
    ax.bar(anual['ano_analise'].astype(str),anual['soma_populacao_disponivel'].astype(float)/1e6,
           color=['#147d92' if a in [2020,2021,2024] else '#c48832' for a in anual['ano_analise']])
    ax.set(ylabel='Milhoes de habitantes',title='Soma das populacoes municipais disponiveis por publicacao')
    imagens.append(grafico(fig,'01_publicacoes','Azul: estimativas. Dourado: Censo 2022 e divulgacao DOU 2023. Diferencas entre publicacoes nao medem diretamente crescimento.'))
    destaque = uf[uf['ano_analise']==ANO_DESTAQUE].sort_values('soma_populacao_disponivel')
    fig,ax=plt.subplots(figsize=(9,8))
    ax.barh(destaque['uf'],destaque['soma_populacao_disponivel'].astype(float)/1e6,color='#147d92')
    ax.set(xlabel='Milhoes de habitantes',title=f'Populacao municipal agregada por UF — {ANO_DESTAQUE}')
    imagens.append(grafico(fig,'02_populacao_uf','Soma dos valores municipais disponiveis. A tabela por UF informa quantos registros nao possuem populacao.'))
    top_atual=top[top['ano_analise']==ANO_DESTAQUE].head(15).sort_values('populacao')
    fig,ax=plt.subplots(figsize=(10,6))
    ax.barh(top_atual['municipio']+' / '+top_atual['uf'],top_atual['populacao'].astype(float)/1e6,color='#147d92')
    ax.set(xlabel='Milhoes de habitantes',title=f'15 maiores municipios — {ANO_DESTAQUE}')
    imagens.append(grafico(fig,'03_maiores_municipios','Ranking de municipios com populacao disponivel no ano selecionado.'))
    p=porte[porte['ano_analise']==ANO_DESTAQUE].set_index('porte').reindex(rotulos).fillna(0)
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(rotulos,p['municipios'],color='#147d92');ax.invert_yaxis()
    ax.set(xlabel='Municipios com populacao disponivel',title=f'Distribuicao por porte populacional — {ANO_DESTAQUE}')
    imagens.append(grafico(fig,'04_porte_municipal','Os intervalos sao definidos pelo script para exploracao; nao constituem classificacao oficial do IBGE.'))
    print('4/4 — Montando o relatorio HTML...',flush=True)
    alertas = diagnostico.groupby('problema').size().reset_index(name='registros')
    aviso = '''<section class="nota"><h2>Leitura dos resultados</h2><p>Cada linha representa um municipio em um ano de analise.
    A populacao e um estoque: nao some os cinco anos. As somas aqui usam apenas valores municipais disponiveis,
    com quantidade de ausencias indicada nas tabelas. Uma ausencia pode decorrer de unidade territorial sem valor
    naquela publicacao; ela nao e substituida por zero.</p><p>2020, 2021 e 2024 usam estimativas municipais.
    2022 usa o Censo. O dado associado a 2023 tambem tem referencia populacional em 2022, com malha territorial de 2023.
    Valores de publicacoes diferentes nao formam uma serie demografica homogenea.
    O arquivo DOU 2023 traz nota sobre ajustes posteriores de limites em Vitoria de Santo Antao e Pombos (PE).</p>
    <p>O comparativo municipal apresenta os valores publicados lado a lado. Nenhuma taxa de crescimento foi calculada.
    Registros estruturais invalidos ou chaves repetidas interrompem a analise para evitar totais duplicados.</p>
    <p>Para o proximo cruzamento: agregar notificacoes por municipio de residencia e ano, depois unir a populacao
    com relacao muitos-para-um validada. Taxas por idade ou sexo exigem denominadores especificos.</p></section>'''
    fontes_links = ''.join('<li>'+html.escape(u)+'</li>' for u in sorted(df['url_fonte'].dropna().unique())) if 'url_fonte' in df else ''
    secoes=[tabela('Fontes e referencias temporais',fontes),tabela('Resumo por publicacao',anual),
        tabela('Alertas de qualidade',alertas),tabela('Perfil das colunas',perfil),
        *imagens,tabela(f'Resumo por regiao — {ANO_DESTAQUE}',regiao[regiao['ano_analise']==ANO_DESTAQUE]),
        tabela(f'20 maiores municipios — {ANO_DESTAQUE}',top[top['ano_analise']==ANO_DESTAQUE]),
        tabela(f'20 menores municipios — {ANO_DESTAQUE}',menores[menores['ano_analise']==ANO_DESTAQUE][['municipio','uf','populacao']]),
        tabela('Concentracao populacional por publicacao',concentracao),
        '<section><h2>Fontes registradas na base</h2><ul>'+fontes_links+'</ul></section>']
    pagina='''<!DOCTYPE html><html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>Observatorio COVID-19 Brasil | Populacao IBGE</title><style>
    body{font-family:Arial,sans-serif;background:#eef3f7;color:#20384a;max-width:1200px;margin:auto;padding:26px}
    h1{font-size:30px}h2{font-size:21px}section{background:white;border-radius:12px;padding:22px;margin:22px 0}
    p{line-height:1.6}.nota{border-left:5px solid #c48832}.scroll{overflow:auto}
    table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:9px;text-align:left;border-bottom:1px solid #dde6ed}
    th{background:#e2edf3}img{width:100%;height:auto}li{overflow-wrap:anywhere;margin:12px 0}
    </style></head><body><h1>Observatorio COVID-19 Brasil</h1><p>Exploracao da populacao municipal IBGE · 2020–2024</p>'''
    pagina += '<p>Gerado em '+html.escape(datetime.now().astimezone().isoformat(timespec='seconds'))+'</p>'
    pagina += aviso+''.join(secoes)+'</body></html>'
    destino=SAIDA/'relatorio_populacao_ibge.html'
    destino.write_text(pagina,encoding='utf-8')
    print(f'\nConcluido. {len(df):,} registros lidos.\nRelatorio: {destino}\nTabelas e graficos: {SAIDA}')
    return {'resumo_anual':anual,'resumo_uf':uf,'resumo_regiao':regiao,'ranking':ranking[campos],
            'alertas':diagnostico,'relatorio':destino}


if __name__=='__main__':
    resultados_populacao=main()


1/4 — Lendo e conferindo a base...
2/4 — Calculando resumos por ano, UF e regiao...
3/4 — Gerando graficos...
4/4 — Montando o relatorio HTML...

Concluido. 27,853 registros lidos.
Relatorio: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_ibge\analise_populacao\relatorio_populacao_ibge.html
Tabelas e graficos: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_ibge\analise_populacao
